In [1]:
import os
import random
import copy
import pandas as pd
import plotly.graph_objects as go
import dash
from dash import dcc, html, ctx
from dash.dependencies import Input, Output, State
import plotly.express as px
from dash import dash_table
import dash_bootstrap_components as dbc
from dash.exceptions import PreventUpdate

from IPython.core.display import display,HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\yakob\AppData\Local\Temp\ipykernel_23928\1043461685.py:14: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display,HTML


In [2]:
csv_path = r'C:\Users\yakob\Documents\Uni\Year3\Block2a\RSinCS\RUG project\data_for_HenkJan.csv' ### change to your path , we will later deliver CSV file with data for you###
labeled_data_path = r'C:\Users\yakob\Documents\Uni\Year3\Block2a\RSinCS\RUG project\labeled_data' ### change to your path, this file is already in the folder with the notebook ###

In [3]:
data = pd.read_csv(csv_path)
data.set_index(['field_node_id', 'timestamp_utc', 'height'], inplace=True)
fnids_list = list(data.index.get_level_values('field_node_id').unique())
labeled_files = os.listdir(labeled_data_path)
if not labeled_files:
    fnids_to_label = fnids_list
else:
    labeled_fnids = [int(i[5:10]) for i in labeled_files]
    fnids_to_label = [fnid for fnid in fnids_list if fnid not in labeled_fnids]

In [4]:
app = dash.Dash(__name__)

x_val_list = []
y_val_list = []


# Define the app layout
app.layout = dbc.Container(
    [
#         html.Div(id='output'),
        dbc.Row(
            dbc.Col(
                html.P("field node id:", 
                      style={'fontSize':20}),
                width={'size': 3, 'offset': 1},  
            )
        ),
        
        dbc.Row([
            dbc.Col(
                dcc.Dropdown(
                    id='fnid-dropdown',
                    options=[{'label': fnid, 'value': fnid} for fnid in fnids_to_label], #fnids_to_label for all fnids and not only the ones I haven't tagged.
                    value=fnids_to_label[0],  # Set the default value
                    searchable=True,
                    style={'width':'80%','font-size': '20px', 'border-radius': '5px'},
                ),
                    width={'size': 3, 'offset': 1},
            ),

             dbc.Col(
                    html.Button(
                        id="save-button",
                        children=["Download to CSV"],
                        style={"fontSize": "20px"}
                    ),
    #                 width=6,
                ),
            dbc.Col(
                    html.Button(
                        id="clear-button",
                        children=["Clear table"],
                        style={"fontSize": "20px"}
                    ),
    #                 width=6,
                    ),
        ]),
    
        dbc.Row(
            dbc.Col(
                dcc.Graph(
                    id='scatter-plot',
                    figure = {},
                    config = {'modeBarButtonsToRemove':['lasso2d', 'select2d', 'toggleSpikelines', 'toImage', 'sendDataToCloud', 'zoomIn2d', 'zoomOut2d'],
                              'modeBarButtonsToAdd': ['pan'], 'scrollZoom': True, 'displayModeBar': True},
                    style={'width': '100%', 'height': '600px'}
                ),
                    width={'size': 12},
            )
        ),
                
        dbc.Row(
            dbc.Col(
                dash_table.DataTable(
                    id='clicked-points-table',
                    columns=[{"name":i,"id":i} for i in ['field_node_id','timestamp_utc','vwc', 'height', 'label']],
                    data=[],
                    row_deletable=True,
                    editable=True,
                    style_table={
                        'fontFamily': 'Arial, sans-serif',
                        'width': '80%',  
                        'height': '300px',  
                        'fontSize': '14px',
                    },
                    style_cell={'textAlign': 'center', 'fontSize': 20},
                    style_header={'textAlign': 'center', 'fontSize': 26, 'fontWeight': 'bold'}
                ),
                 width={'size': 12, 'offset': 1}
            )
        ),

    dcc.Store(id='clicked-points-store', data=[]),
    dcc.Download(id="download-button"),
    
        ], fluid=True)
    
  

@app.callback(
    Output('scatter-plot', 'figure'),
#     Output("output", "children"),
    Input('fnid-dropdown', 'value'),
    Input('scatter-plot', 'clickData'),
    State('scatter-plot', 'relayoutData')
)

def update_scatter_plot(selected_fnid, click_data, relayout_data):
    global x_val_list, y_val_list
    
    trigger_id  = ctx.triggered_id
    
    filtered_data = data.loc[selected_fnid,:].reset_index()
    
    fig = px.line(filtered_data,
                  x = 'timestamp_utc',
                  y = 'vwc',
                  color = 'height',
                  template="plotly_white",
                  custom_data = ['height'],
                  markers = True)

    
    fig.update_layout(
        xaxis=dict(gridwidth=4, tickfont=dict(size=18)),
        xaxis_title='Timestamp_UTC',
        xaxis_title_font = dict(size=20),
        yaxis=dict(gridwidth=4, tickfont=dict(size=18)),
        yaxis_title='VWC (%)',
        yaxis_title_font = dict(size=20),
        legend=dict(title="Height", font=dict(size=18)),
        dragmode="pan")
    
#     fig.update_traces(customdata=filtered_data['height'])
    
#     fig.update_traces(mode="markers+lines")
    
    if trigger_id == 'scatter-plot':
     
        
        x_val = click_data['points'][0]['x']
        y_val = click_data['points'][0]['y']
        
        
        if (x_val in x_val_list) and (y_val in y_val_list):
            x_val_list.remove(x_val)
            y_val_list.remove(y_val)
        
        else:
        
            x_val_list.append(x_val)
            y_val_list.append(y_val)

        # Update the clicked point in the scatter plot
        fig.add_trace(go.Scatter(
            x=x_val_list,
            y=y_val_list,
            mode='markers',
            marker=dict(size=15, color='black', symbol='x'), #line=dict(width=4)
            name='Clicked Point'
        ))
        
    if trigger_id == 'fnid-dropdown':
        x_val_list.clear()
        y_val_list.clear()
        
    if relayout_data:
        fig = copy.deepcopy(fig)
        if 'xaxis.range[0]' in relayout_data:
            fig['layout']['xaxis']['range'] = [
                relayout_data['xaxis.range[0]'],
                relayout_data['xaxis.range[1]']
            ]
        if 'yaxis.range[0]' in relayout_data:
            fig['layout']['yaxis']['range'] = [
                relayout_data['yaxis.range[0]'],
                relayout_data['yaxis.range[1]']
            ]            
  
    return fig


@app.callback(
#     Output("output", "children"),
    Output('clicked-points-store', 'data'),
    Input('fnid-dropdown', 'value'),
    Input('scatter-plot', 'clickData')
)

def storing_data(selected_fnid, click_data):
    trigger_id  = ctx.triggered_id
    if trigger_id == 'scatter-plot':
#     if clickData is not None:
        timestamp = click_data['points'][0]['x']
        vwc = click_data['points'][0]['y']
#         n = json.dumps(click_data)
        if 'customdata' in click_data['points'][0].keys():
            height = click_data['points'][0]['customdata'][0]
        else: 
            height = -1
        
    else:
        raise PreventUpdate  
        
    return [selected_fnid, timestamp, round(vwc, 3), height, 1]



@app.callback(
    Output('clicked-points-table', 'data'),
    Output("clicked-points-table", "columns"),
#     Output("output", "children"),
    Input('clicked-points-store', 'data'), 
    State('fnid-dropdown', 'value'), 
    State("clicked-points-table","data"),
    Input('scatter-plot', 'clickData'),
    Input("clear-button", "n_clicks"),
    prevent_initial_call=True
)


def update_table(stored_data, selectd_fnid, table_data, click_data, clear_button):
    trigger_id  = ctx.triggered_id
    
    df_table = pd.DataFrame(table_data, columns=['field_node_id','timestamp_utc', 'vwc', 'height', 'label'])
    
    if trigger_id == 'scatter-plot':
        
        timestamp = stored_data[1]
        if timestamp not in df_table['timestamp_utc'].values:

            df_table.loc[-1] = stored_data
            df_table.index = df_table.index + 1 # so every new row will present as the first row
            df_table.sort_index(inplace=True)
#         df_table.drop_duplicates(inplace=True)

#         point_data = str(stored_data)
        df_data = df_table.to_dict(orient='records')
        df_columns = [{"name": i, "id": i} for i in df_table.columns]
        
    if trigger_id == 'clear-button':
        df_data = []
        df_columns = [{"name": i, "id": i} for i in ['field_node_id','timestamp_utc', 'vwc', 'height', 'label']]
        
#     else:
#         raise PreventUpdate
        
    return  (df_data, df_columns)


@app.callback(
    Output("download-button", "data"),
    State('fnid-dropdown', 'value'),
    Input("save-button", "n_clicks"),
    Input('clicked-points-table', 'data'),
    
)
def save_table(selected_fnid, n_clicks, table_data):
    prevent_initial_call = True
    trigger_id  = ctx.triggered_id
    if trigger_id == 'save-button':
#     if n_clicks:
        df_to_save = pd.DataFrame(table_data, columns = ['field_node_id','timestamp_utc', 'vwc', 'height', 'label'])
        file_name = f"fnid_{selected_fnid}_labeling.csv"
        
        return dcc.send_data_frame(df_to_save.to_csv, file_name, index=False)
  
        
    
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="tab")  
#     app.run()

Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>